# Lab 8: Support functions and minimum width
**Python supplement · Student notebook · English**

Compare constant and minimum width; normalize convex polygons correctly and test the bounds behind the existence proof.

These two exercises supplement the mathematical lab; they are not
additional required work inside its original 90-minute schedule.
Allow roughly 60–90 minutes, depending on Python experience.

Run the setup cell first, then work from top to bottom. Functions
marked TODO return `None` until completed; the demonstration cells
will tell you what to finish. After each edit, rerun the definition
and its demonstration. `assert` checks then test useful mathematical
properties. Write your explanations in the response cells.

A plot, finite search or successful iteration supplies numerical
evidence. State separately any theorem used to establish existence,
global optimality or an equality case.


Prerequisites: basic Python variables, loops, functions and NumPy arrays. The supplied plotting and geometry helpers can be used without reimplementing them. See [setup instructions](README.md).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.figsize': (9, 4), 'font.size': 11,
                     'axes.grid': True, 'grid.alpha': 0.25,
                     'figure.constrained_layout.use': True})

def closed(v):
    """Repeat the first vertex only for plotting."""
    return np.vstack([v, v[0]])

def area(v):
    """Unsigned shoelace area of an ordered simple polygon."""
    return abs(np.sum(v[:, 0]*np.roll(v[:, 1], -1)
                      - v[:, 1]*np.roll(v[:, 0], -1)))/2

def perimeter(v):
    return np.linalg.norm(np.roll(v, -1, axis=0)-v, axis=1).sum()

def outline(ax, v, **kwargs):
    p = closed(v)
    ax.plot(p[:, 0], p[:, 1], **kwargs)
    ax.set_aspect('equal', adjustable='box')

from scipy.spatial import ConvexHull, QhullError


## Exercise 8.1: Width curves of three shapes

For vertices $v_i$ representing a convex polygon, compute
$h(\theta)=\max_i v_i\cdot u_\theta$ and
$w(\theta)=\max_i v_i\cdot u_\theta-\min_i v_i\cdot u_\theta$,
where $u_\theta=(\cos\theta,\sin\theta)$.

**Code tasks (30–40 minutes).**
1. Complete `support_width(vertices, angles)`, returning arrays of sampled
   support values and widths. Use a matrix product for all projections.
2. Plot width curves for a disk of diameter 1, a Reuleaux triangle of width 1,
   and an equilateral triangle of altitude 1. Their sampled boundaries are supplied.
3. Translate the triangle by $(2,-1)$. Check that support changes by
   $2\cos\theta-\sin\theta$ but width is unchanged; display the signed supports.

**Return contract:** `(support, width)`. All three shapes have minimum width 1,
but the equilateral triangle is not of constant width. Sampled curved boundaries
represent inscribed polygons, so expect a small geometric discretization error.
Refining the boundary and refining the angular grid are different operations.


**Provided helper code.** Run this cell before implementing the task.

In [ ]:
def width_examples(samples_per_arc=120):
    t=np.linspace(0,2*np.pi,3*samples_per_arc,endpoint=False)
    disk=.5*np.column_stack([np.cos(t),np.sin(t)])
    pieces=[]
    for center,start in [([.5,np.sqrt(3)/2],240),([0.,0.],0),([1.,0.],120)]:
        theta=np.radians(np.linspace(start,start+60,samples_per_arc,endpoint=False))
        pieces.append(np.array(center)+np.column_stack([np.cos(theta),np.sin(theta)]))
    reuleaux=np.vstack(pieces)
    triangle=np.array([[0.,0.],[2/np.sqrt(3),0.],[1/np.sqrt(3),1.]])
    return {'disk':disk,'Reuleaux triangle':reuleaux,'equilateral triangle':triangle}


In [ ]:
def support_width(vertices,angles):
    # TODO: construct the unit direction matrix, project all vertices,
    # and compute column-wise maxima and max-minus-min.
    return None


**Run, visualize and check.** Adapt these plots as requested, and report what changes when you refine the discretization or vary parameters.

In [ ]:
shapes=width_examples(120)
angles=np.linspace(0,2*np.pi,721)
result=support_width(shapes['equilateral triangle'],angles)
if result is None:
    print('Complete support_width, then rerun this cell.')
else:
    fig,ax=plt.subplots(1,3,figsize=(14,4))
    for name,v in shapes.items():
        h,w=support_width(v,angles)
        outline(ax[0],v-v.mean(axis=0),label=name)
        ax[1].plot(np.degrees(angles),w,label=name)
        if name != 'equilateral triangle':
            np.testing.assert_allclose(w,1.,atol=2e-4)
        print(name,': sampled width range',w.min(),w.max())
    triangle=shapes['equilateral triangle']
    h,w=support_width(triangle,angles)
    shifted_h,shifted_w=support_width(triangle+np.array([2.,-1.]),angles)
    np.testing.assert_allclose(shifted_w,w,atol=1e-12)
    np.testing.assert_allclose(shifted_h-h,2*np.cos(angles)-np.sin(angles),atol=1e-12)
    assert shifted_h.min() < 0
    ax[0].legend(fontsize=8); ax[0].set_title('Centered only for this display')
    ax[1].set(xlabel='normal angle (degrees)',ylabel='directional width'); ax[1].legend(fontsize=8)
    ax[2].plot(np.degrees(angles),h,label='original triangle')
    ax[2].plot(np.degrees(angles),shifted_h,label='translated triangle')
    ax[2].axhline(0,color='gray'); ax[2].set(xlabel='normal angle (degrees)',ylabel='signed support')
    ax[2].legend(fontsize=8); plt.show()


**Explain your observations.** Why can support be negative? Which shape features would be lost if a nonconvex point set were replaced by its support function?

*Write your response here.*

## Exercise 8.2: Search convex polygons at minimum width one

Generate random planar point sets, take their convex hulls, and scale each
result to **true polygonal minimum width** 1. The minimum width of a convex
polygon occurs in a direction normal to an edge; use the supplied exact
finite-direction helper rather than an unrelated uniform angle grid.

**Code tasks (35–45 minutes).**
1. Complete `normalized_polygons(count, seed)`: for each sample, draw
   between 3 and 12 Gaussian points, obtain the counterclockwise hull,
   center it and divide coordinates by its minimum width.
2. Return the polygons and a table with columns `(area, diameter, minimum_width)`.
   Plot area against diameter and draw the best random candidate next to
   the exact equilateral triangle of altitude 1.
3. Check $A\geq D/2$ and Pál's bound $A\geq1/\sqrt3$.
   Repeat with another seed and sample count. Explain why the best random
   candidate does not normally achieve the equality case exactly.

**Return contract:** `(polygons, measurements)` with a list of `count`
vertex arrays and a `(count,3)` NumPy array. Use `np.random.default_rng(seed)`
inside the function so rerunning a cell is reproducible. Reject degenerate
hulls or widths below $10^{-8}$, with a finite attempt limit.

SciPy's `ConvexHull.vertices` is counterclockwise in 2D. Beware of names:
`hull.area` is perimeter in 2D and `hull.volume` is area. We use the supplied
shoelace function for area to keep the convention explicit.


**Provided helper code.** Run this cell before implementing the task.

In [ ]:
def polygon_min_width(v):
    edges=np.roll(v,-1,axis=0)-v
    lengths=np.linalg.norm(edges,axis=1)
    if np.any(lengths < 1e-14):
        raise ValueError('Distinct consecutive vertices are required.')
    normals=np.column_stack([-edges[:,1],edges[:,0]])/lengths[:,None]
    return np.ptp(v @ normals.T,axis=0).min()

def polygon_diameter(v):
    return np.linalg.norm(v[:,None,:]-v[None,:,:],axis=2).max()


In [ ]:
def normalized_polygons(count=400,seed=8):
    # TODO: generate points, form hulls, skip degeneracy, normalize width.
    # TODO: store polygon vertices and (area,diameter,width) measurements.
    return None


**Run, visualize and check.** Adapt these plots as requested, and report what changes when you refine the discretization or vary parameters.

In [ ]:
result=normalized_polygons()
if result is None:
    print('Complete normalized_polygons, then rerun this cell.')
else:
    polygons,table=result
    areas,diameters,widths=table.T
    np.testing.assert_allclose(widths,1.,atol=1e-10)
    assert np.all(areas >= diameters/2-1e-10)
    assert np.all(areas >= 1/np.sqrt(3)-1e-10)
    triangle=width_examples()['equilateral triangle']
    np.testing.assert_allclose(polygon_min_width(triangle),1.)
    np.testing.assert_allclose(area(triangle),1/np.sqrt(3))
    k=areas.argmin()
    fig,ax=plt.subplots(1,3,figsize=(14,4))
    ax[0].scatter(diameters,areas,s=12,alpha=.5,label='random convex polygons')
    grid=np.geomspace(min(diameters.min(),1.),diameters.max(),200)
    ax[0].plot(grid,grid/2,label='area-diameter lower bound')
    ax[0].axhline(1/np.sqrt(3),color='orange',label='Pal lower bound')
    ax[0].set(xscale='log',yscale='log',xlabel='diameter at width 1',ylabel='area at width 1')
    ax[0].legend(fontsize=8)
    outline(ax[1],polygons[k],label='best random candidate')
    outline(ax[2],triangle-triangle.mean(axis=0),color='orange',label='exact optimal triangle')
    for panel in ax[1:]:
        panel.set(xlim=(-.8,.8),ylim=(-.8,.8)); panel.legend(fontsize=8)
    plt.show()
    print('Best random area:',areas[k],'; theoretical minimum:',1/np.sqrt(3))
    # A rectangle tests the edge-normal minimum independently of an angle grid.
    rectangle=np.array([[0.,0.],[5.,0.],[5.,2.],[0.,2.]])
    angle=.173
    rotation=np.array([[np.cos(angle),-np.sin(angle)],[np.sin(angle),np.cos(angle)]])
    np.testing.assert_allclose(polygon_min_width(rectangle @ rotation.T),2.)
    first=normalized_polygons(5,19)[1]
    np.testing.assert_allclose(first,normalized_polygons(5,19)[1])


**Explain your observations.** Why does fixed minimum width not bound diameter? How does restricting area to a sublevel restore confinement, and why does a random search not prove existence?

*Write your response here.*

**Before submitting:** restart the kernel, run all cells in order, check axis labels and units, and explain any discrepancy between numerical and exact quantities.